# 🌱 SAGE Inference Server (Kaggle / Colab GPU)

Serves **Qwen2.5-VL-3B-Instruct + per-crop LoRA adapters** over a public Cloudflare tunnel,
exposing the API the SAGE middleware expects:

| Method | Path | Purpose |
|---|---|---|
| GET | `/health` | liveness + active crop |
| GET | `/api/v1/crops` | crops, default prompts, known classes |
| GET | `/api/v1/model/status` | adapters + GPU memory |
| POST | `/api/v1/predict` | `crop`, `image` (`prompt`, `max_new_tokens`, `temperature` are accepted but ignored) |

Adapters are downloaded **on demand** the first time the middleware asks for a crop, then kept in GPU memory,
so switching crops is instant after the first request.

Prompts, labels and output parsing come from `sage_train/` (TejaAnish/sage), pinned to the training code. The model sees **only the photo and the crop**. It gives no confidence score, has no healthy class, and `class_name` is `null` when its output can't be parsed.

**Before running:** Settings → Accelerator **GPU T4 x2** (or P100), Internet **On**,
and add a secret named **`HF_TOKEN`** (Add-ons → Secrets). Optional secret **`SAGE_API_KEY`** to protect the API.
Then **Run All** and copy the `BACKEND_URL=` line printed at the end into `middleware/.env`.

In [ ]:
# 1. Install the versions used for training (≈1–2 min). Needed in every new Kaggle session; keep the pins.
!pip install -q transformers==4.57.1 peft==0.17.1 accelerate==1.10.1 datasets==4.1.1 huggingface_hub==0.35.3 "pillow<13" fastapi "uvicorn[standard]" python-multipart

In [ ]:
# 2. Configuration
import os, re, glob

# Adapter locations (from modelpath.txt), as "<repo>:<subfolder>".
MODELPATH_TXT = """
corn - tejaanish/sage-qwen25vl3b-corn-lora:checkpoints/last
rice - tejaanish/sage-qwen25vl3b-rice-lora:checkpoints/last
soybean - tejaanish/sage-qwen25vl3b-soybean-lora:checkpoints/last
tomato - tejaanish/sage-qwen25vl3b-tomato-lora:checkpoints/last
wheat - tejaanish/sage-qwen25vl3b-wheat-lora:checkpoints/last
"""

BASE_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
ADAPTER_CACHE_DIR = f"{WORK_DIR}/sage_adapters"
SAGE_TRAIN_REPO = "https://github.com/TejaAnish/sage"
SAGE_TRAIN_COMMIT = "8ad2951b50221eabddfb5c47c49174de2922a3e1"  # prompts must match the training text exactly
MIN_PIXELS, MAX_PIXELS = 224 * 224, 448 * 448  # training image limits
PORT = 8000
MAX_IMAGE_MB = 10
PRELOAD_CROPS = []           # e.g. ["tomato"] to download + attach adapters at startup


def parse_modelpath(text):
    """Returns {crop: (repo, subfolder)}."""
    base, entries = None, {}
    for line in text.strip().splitlines():
        if " - " not in line:
            continue
        key, value = (s.strip() for s in line.split(" - ", 1))
        key = key.lower()
        if key == "base":
            base = value
        elif key != "url":
            entries[key] = value
    adapters = {}
    for crop, value in entries.items():
        repo, sep, folder = value.partition(":")
        adapters[crop] = (repo, folder.strip("/")) if sep else (base, value.strip("/"))
    return adapters


# Prefer a modelpath.txt uploaded next to the notebook / as a dataset, else the inline copy above.
_mp_files = glob.glob("modelpath.txt") + glob.glob("/kaggle/input/**/modelpath.txt", recursive=True)
if _mp_files:
    MODELPATH_TXT = open(_mp_files[0]).read()
    print("Using", _mp_files[0])
ADAPTERS = parse_modelpath(MODELPATH_TXT)


def load_secret(name):
    """Kaggle Secrets → Colab userdata → environment → any .env file nearby."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name]
    for f in glob.glob(".env") + glob.glob("/kaggle/input/**/.env", recursive=True):
        for line in open(f):
            k, _, v = line.strip().partition("=")
            if k.strip() == name and v.strip():
                return v.strip().strip('"\'')
    return None


HF_TOKEN = load_secret("HF_TOKEN")
SAGE_API_KEY = load_secret("SAGE_API_KEY") or ""
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

print("Adapters     :", ADAPTERS)
print("HF_TOKEN     :", "found" if HF_TOKEN else "MISSING (public/slow downloads only)")
print("API key auth :", "enabled" if SAGE_API_KEY else "disabled")

In [ ]:
# 3. Crop registry, prompts and parser: the training code is the source of truth. Don't rewrite these.
import subprocess, sys

SAGE_CODE_DIR = f"{WORK_DIR}/sage_code"
if not os.path.isdir(f"{SAGE_CODE_DIR}/sage_train"):
    subprocess.run(["git", "clone", "-q", SAGE_TRAIN_REPO, SAGE_CODE_DIR], check=True)
    subprocess.run(["git", "-C", SAGE_CODE_DIR, "checkout", "-q", SAGE_TRAIN_COMMIT], check=True)
sys.path.insert(0, SAGE_CODE_DIR)

from sage_train.catalog import CROPS as CATALOG, CROP_ALIASES, disease_names, resolve_crop
from sage_train.kaggle_env import discover_model, prepare_processor_files
from sage_train.prompts import build_messages, parse_prediction, prediction_details, user_prompt

CROP_CLASSES = {c: disease_names(c) for c in CATALOG}
DISPLAY = {c: CATALOG[c]["display"] for c in CATALOG}
CROPS = [c for c in CATALOG if c in ADAPTERS]
unknown = set(ADAPTERS) - set(CATALOG)
assert not unknown, f"Not in sage_train/catalog.py: {unknown}"


def served_crop(name):
    """Resolve aliases (maize, paddy) to a crop with an adapter, or raise ValueError."""
    supported = ", ".join(CROPS + [a for a, c in CROP_ALIASES.items() if c in CROPS])
    try:
        crop = resolve_crop(name)
    except KeyError:
        raise ValueError(f"Unsupported crop '{name}'. Supported: {supported}")
    if crop not in ADAPTERS:
        raise ValueError(f"Unsupported crop '{name}'. Supported: {supported}")
    return crop


print("Serving crops:", CROPS)

In [ ]:
# 4. Load processor + base model
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

assert torch.cuda.is_available(), "Enable a GPU accelerator in the notebook settings!"
print("GPU:", torch.cuda.get_device_name(0), "| count:", torch.cuda.device_count())

try:
    MODEL_PATH = discover_model()  # the Qwen2.5-VL model attached as a Kaggle input
except SystemExit as e:
    print(e, "\nDownloading from the HF Hub instead.")
    from huggingface_hub import snapshot_download
    MODEL_PATH = snapshot_download(BASE_MODEL_ID, token=HF_TOKEN)

# Required: corrects the image processor class name and sets the training pixel limits.
processor = AutoProcessor.from_pretrained(
    prepare_processor_files(MODEL_PATH, f"{WORK_DIR}/processor_files", MIN_PIXELS, MAX_PIXELS))
processor.tokenizer.padding_side = "left"
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, device_map={"": 0}, attn_implementation="sdpa",
)
base_model.eval()
print("Base model loaded from", MODEL_PATH,
      "| GPU mem:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

In [ ]:
# 5. On-demand adapter manager
import threading, time
from huggingface_hub import snapshot_download
from peft import PeftModel

GPU_LOCK = threading.RLock()   # one generation / adapter switch at a time


class AdapterManager:
    def __init__(self, base):
        self.base = base
        self.model = None            # PeftModel once the first adapter is attached
        self.active = None
        self.loaded = []
        self.status = {c: "not_loaded" for c in CROPS}

    def local_path(self, crop):
        repo, folder = ADAPTERS[crop]
        root = f"{ADAPTER_CACHE_DIR}/{crop}"
        # adapter_* only: training checkpoints also hold a large optimizer.pt that inference doesn't need.
        snapshot_download(repo_id=repo, allow_patterns=[f"{folder}/adapter_*" if folder else "adapter_*"],
                          local_dir=root, token=HF_TOKEN)
        path = os.path.join(root, folder)
        assert os.path.isfile(os.path.join(path, "adapter_config.json")), f"adapter_config.json missing in {path}"
        return path

    def activate(self, crop):
        """Download (first time), attach, and switch to the crop's adapter. Caller holds GPU_LOCK."""
        if crop == self.active:
            return
        if crop not in self.loaded:
            self.status[crop] = "loading"
            t = time.time()
            try:
                path = self.local_path(crop)
                if self.model is None:
                    self.model = PeftModel.from_pretrained(self.base, path, adapter_name=crop, is_trainable=False)
                else:
                    self.model.load_adapter(path, adapter_name=crop, is_trainable=False)
            except Exception:
                self.status[crop] = "error"
                raise
            self.loaded.append(crop)
            self.status[crop] = "loaded"
            print(f"[adapter] {crop} attached in {time.time() - t:.1f}s")
        self.model.set_adapter(crop)
        self.model.eval()
        self.active = crop
        print(f"[adapter] active = {crop}")

    def runtime(self):
        return self.model if self.model is not None else self.base


adapters = AdapterManager(base_model)
for c in PRELOAD_CROPS:
    with GPU_LOCK:
        adapters.activate(c)

In [ ]:
# 6. Inference: same prompt, generation settings and parser as training evaluation
import io
from PIL import Image, ImageOps


def load_image(data):
    with Image.open(io.BytesIO(data)) as im:
        return ImageOps.exif_transpose(im).convert("RGB")


def predict(crop, image_bytes):
    with Image.open(io.BytesIO(image_bytes)) as probe:
        fmt, (w, h) = probe.format, probe.size
    image = load_image(image_bytes)
    text = processor.apply_chat_template(build_messages(crop), tokenize=False, add_generation_prompt=True)
    batch = processor(text=[text], images=[image], padding=True, return_tensors="pt").to("cuda")

    with GPU_LOCK:  # the active adapter is global, so switching and generating must not overlap
        adapters.activate(crop)
        t = time.time()
        with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
            out = adapters.runtime().generate(**batch, max_new_tokens=16, do_sample=False, use_cache=True,
                                              temperature=None, top_p=None, top_k=None,
                                              pad_token_id=processor.tokenizer.pad_token_id)
        elapsed = time.time() - t

    n_in = batch["input_ids"].shape[1]
    raw = processor.tokenizer.decode(out[0, n_in:], skip_special_tokens=True).strip()
    disease = parse_prediction(raw, disease_names(crop))  # None = output couldn't be parsed
    details = prediction_details(crop, disease) if disease else {}
    repo, folder = ADAPTERS[crop]
    return {
        "success": True,
        "crop": crop,
        "base_model": BASE_MODEL_ID,
        "active_adapter": crop,
        "checkpoint": f"{repo}/{folder}",
        "input": {"prompt": user_prompt(crop)},
        "prediction": {
            "response": raw,
            "class_name": disease,
            "crop": crop,
            "disease_type": details.get("type"), "plant_organ": None, "visual_symptoms": None,
            "pathogen": details.get("pathogen"),
            "matched_taxonomy_class": disease,
            "is_canonical_match": disease is not None,
            "attributes": {"generated_tokens": int(out.shape[1] - n_in)},
        },
        "raw_output": raw,
        "inference_time_seconds": round(elapsed, 3),
        "image_metadata": {"width": w, "height": h, "format": fmt},
    }

In [ ]:
# 7. FastAPI app (same contract the middleware already uses)
from typing import Optional
from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile

ALLOWED = {"image/jpeg", "image/jpg", "image/png", "image/webp"}
app = FastAPI(title="SAGE Inference Server")


def check_key(authorization: Optional[str], x_api_key: Optional[str]):
    if not SAGE_API_KEY:
        return
    bearer = (authorization or "").removeprefix("Bearer ").strip()
    if SAGE_API_KEY not in (bearer, (x_api_key or "").strip()):
        raise HTTPException(401, "Invalid or missing API key.")


def gpu_status():
    if not torch.cuda.is_available():
        return {"available": False}
    gb = 1024 ** 3
    return {"available": True, "device_name": torch.cuda.get_device_name(0),
            "total_memory_gb": round(sum(torch.cuda.get_device_properties(i).total_memory for i in range(torch.cuda.device_count())) / gb, 2),
            "allocated_memory_gb": round(sum(torch.cuda.memory_allocated(i) for i in range(torch.cuda.device_count())) / gb, 2),
            "reserved_memory_gb": round(sum(torch.cuda.memory_reserved(i) for i in range(torch.cuda.device_count())) / gb, 2)}


@app.get("/health")
def health():
    return {"status": "healthy", "service": "SAGE API", "device": "cuda", "base_model_loaded": True,
            "active_crop": adapters.active}


@app.get("/api/v1/crops")
def crops(authorization: Optional[str] = Header(None), x_api_key: Optional[str] = Header(None)):
    check_key(authorization, x_api_key)
    details = [{
        "crop_id": c, "display_name": DISPLAY.get(c, c.title()), "hf_repo": ADAPTERS[c][0],
        "checkpoint_subpath": ADAPTERS[c][1], "model_type": "peft_lora", "adapter_name": c,
        "default_prompt": user_prompt(c), "loading_status": adapters.status[c],
        "classes_count": len(CROP_CLASSES[c]), "known_classes": CROP_CLASSES[c],
    } for c in CROPS]
    return {"total_crops": len(CROPS), "available_crops": CROPS, "crop_details": details}


@app.get("/api/v1/model/status")
def model_status(authorization: Optional[str] = Header(None), x_api_key: Optional[str] = Header(None)):
    check_key(authorization, x_api_key)
    return {"base_model": BASE_MODEL_ID, "active_crop": adapters.active, "active_adapter": adapters.active,
            "adapter_mode": "peft_multi_adapter", "base_model_loaded": True, "loaded_adapters": adapters.loaded,
            "gpu_status": gpu_status(), "mock_mode": False}


@app.post("/api/v1/predict")
def predict_route(crop: str = Form(...), image: UploadFile = File(...), prompt: Optional[str] = Form(None),
                  max_new_tokens: Optional[int] = Form(None), temperature: Optional[float] = Form(None),
                  authorization: Optional[str] = Header(None), x_api_key: Optional[str] = Header(None)):
    check_key(authorization, x_api_key)
    # prompt, max_new_tokens and temperature stay in the signature for the existing contract but are ignored:
    # the adapters were trained on one exact prompt, and nothing but the photo and crop may affect a diagnosis.
    try:
        crop_key = served_crop(crop)
    except ValueError as e:
        raise HTTPException(400, str(e))
    if (image.content_type or "").lower() not in ALLOWED:
        raise HTTPException(415, f"Unsupported image type '{image.content_type}'. Use JPEG, PNG or WebP.")
    data = image.file.read()
    if not data:
        raise HTTPException(400, "Uploaded image is empty.")
    if len(data) > MAX_IMAGE_MB * 1024 * 1024:
        raise HTTPException(413, f"Image exceeds {MAX_IMAGE_MB} MB.")
    try:
        Image.open(io.BytesIO(data)).verify()
    except Exception:
        raise HTTPException(422, "Uploaded file is corrupted or not a valid image.")
    try:
        return predict(crop_key, data)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        raise HTTPException(500, "GPU out of memory — try again.")
    except Exception as e:
        raise HTTPException(500, f"Inference failed: {e}")

In [ ]:
# 8. Start the server + public Cloudflare tunnel
import subprocess, uvicorn, urllib.request, json as _json

server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
def http_get(url, headers=None, timeout=15):
    req = urllib.request.Request(url, headers={"User-Agent": "sage-notebook", **(headers or {})})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.status, _json.loads(r.read())

for _ in range(60):
    try:
        http_get(f"http://127.0.0.1:{PORT}/health", timeout=2)
        break
    except Exception as e:
        last_err = e
        time.sleep(1)
else:
    raise RuntimeError(f"Local server did not start: {last_err!r}")
print("Local server is up on port", PORT)

if not os.path.exists("cloudflared"):
    subprocess.run(["wget", "-q", "-O", "cloudflared",
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    os.chmod("cloudflared", 0o755)

tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
PUBLIC_URL = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        PUBLIC_URL = m.group(0)
        break
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()  # keep draining logs

print("\n" + "=" * 70)
print("  SAGE backend is live:", PUBLIC_URL)
print("  Paste into middleware/.env and restart the middleware:\n")
print(f"  BACKEND_URL={PUBLIC_URL}")
if SAGE_API_KEY:
    print("  BACKEND_API_KEY=<the SAGE_API_KEY secret you set>")
print("=" * 70)

In [ ]:
# 9. Smoke test through the public URL (tunnel DNS can take ~10–30 s to propagate)
hdr = {"Authorization": f"Bearer {SAGE_API_KEY}"} if SAGE_API_KEY else {}
for attempt in range(10):
    try:
        status, body = http_get(f"{PUBLIC_URL}/api/v1/model/status", headers=hdr)
        print("Tunnel OK:", status, body)
        break
    except Exception as e:
        print("waiting for tunnel…", repr(e)[:200])
        time.sleep(5)

In [ ]:
# 10. Keep the session alive and log status every 5 min (interrupt this cell to stop logging; the server keeps running)
try:
    while True:
        g = gpu_status()
        print(time.strftime("%H:%M:%S"), "| active:", adapters.active, "| loaded:", adapters.loaded,
              "| GPU alloc:", g.get("allocated_memory_gb"), "GB |", PUBLIC_URL)
        time.sleep(300)
except KeyboardInterrupt:
    print("Logging stopped; server still running.")